# Maize data cleaning with seasonal satellite medians

**Felix Yu · DSC 291 DSSD**

| | |
| --- | --- |
| Input | `data/team_data_full_v1.duckdb` |
| Output | `data/clean/felix_seasonal_contrast.parquet` |
| Unit | One row per unlocked maize field-season, with yield in tonnes per hectare |
| Run | Use Python 3.12 and the repository’s `requirements.txt`. Run sections 1–5 to produce the clean table. The Ridge check follows the export. |

**Cleaning rationale.** A single clear image can capture an unusual day. I use the median of the clear observations in each month and keep a field even if clouds leave holes in its record. When both years have usable observations, I measure how the field’s canopy changed from the same month last year. The number of matched months tells the next model how much evidence supports each comparison. Weather, soil and terrain provide context that remains available for many fields with sparse imagery.

**Scope.** The inventory covers all 31 country-years in the release. The project’s label and spatial rules yield 25,579 development field-seasons in 22 country-years across eight countries. This table describes conditions through the end of the season.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import duckdb
import numpy as np
import pandas as pd

db_path = next(
    (base / "data/team_data_full_v1.duckdb" for base in (Path.cwd(), *Path.cwd().parents)
     if (base / "data/team_data_full_v1.duckdb").is_file()),
    None,
)
if db_path is None:
    raise FileNotFoundError("Place team_data_full_v1.duckdb in the repository's data/ folder.")

db = duckdb.connect(str(db_path), read_only=True)
out_path = db_path.parent / "clean/felix_seasonal_contrast.parquet"
gaps = [f"gap_{k}" for k in range(1, 5)]

## 1. Audit the labels and apply the project’s spatial folds

| Decision | How it affects the clean table |
| --- | --- |
| Inherit the release’s maize `keep` rule | Retain eligible field locations and measured yields in t/ha, including eligible measured zeros. The inventory lists reasons for excluded maize rows by country and year. |
| Join the official `folds_all` table | Restrict the export to unlocked fields in folds 1–4. Keep each fold’s 2 km buffer flag for training. |
| Keep eligible fields with missing imagery | Leave missing measurements empty so every cleaning alternative can be scored on the same fields. |

The first table counts every source record, including other crops. The `development` column is the number of rows from that country-year that enter the export. Nigeria’s 115 maize records have unusable locations, which accounts for its empty development group.

In [2]:
inventory = db.sql("""
    SELECT country_iso3, season_year, count(*) AS source_rows,
           count(*) FILTER (WHERE crop = 'maize') AS maize, count(*) FILTER (WHERE keep) AS kept,
           count(*) FILTER (WHERE keep AND f.locked) AS sealed,
           count(*) FILTER (WHERE keep AND NOT f.locked AND fold BETWEEN 1 AND 4) AS development,
           string_agg(DISTINCT exclusion_reason, ', ' ORDER BY exclusion_reason)
               FILTER (WHERE NOT keep AND crop = 'maize') AS excluded_maize_reasons
    FROM labels LEFT JOIN folds_all f USING (release, field_season_id)
    GROUP BY ALL ORDER BY country_iso3, season_year
""").df()
assert inventory.source_rows.sum() == db.sql("SELECT count(*) FROM labels").fetchone()[0]
assert (inventory.kept == inventory.sealed + inventory.development).all()
print(inventory.fillna({"excluded_maize_reasons": ""}).to_string(index=False))

country_iso3  season_year  source_rows  maize  kept  sealed  development                   excluded_maize_reasons
         BDI         2015         4316    610     0       0            0                      location_not_usable
         BDI         2016         4242   1383     0       0            0                      location_not_usable
         BDI         2017         4906   1430  1052     221          831                      location_not_usable
         BDI         2019         6288   1686     0       0            0                      location_not_usable
         BDI         2020         4708   1268     0       0            0                      location_not_usable
         KEN         2016         3934   3934  3770    1522         2248                      location_not_usable
         KEN         2017         4002   4002  3991    1549         2442                      location_not_usable
         KEN         2018         4280   4280  4231    1375         2856                

In [3]:
db.execute("""
    CREATE OR REPLACE TEMP VIEW development AS
    SELECT release, field_season_id, country_iso3, data_set, season_year, yield_t_ha,
           fold, locked, gap_1, gap_2, gap_3, gap_4
    FROM labels JOIN folds_all USING (release, field_season_id)
    WHERE keep AND crop = 'maize' AND NOT locked AND fold BETWEEN 1 AND 4
""")
def query_fields(sql):
    frame = db.sql(sql).df().set_index("field_season_id")
    assert frame.index.is_unique
    return frame

fields = query_fields("SELECT * FROM development ORDER BY field_season_id")
assert len(fields) == inventory.development.sum()
assert not fields.locked.any() and fields[gaps].notna().all().all()
assert np.isfinite(fields.yield_t_ha).all() and (fields.yield_t_ha >= 0).all()
print(f"{len(fields):,} development rows across {fields.country_iso3.nunique()} countries "
      f"and {fields.data_set.nunique()} source datasets.")

25,579 development rows across 8 countries and 11 source datasets.


## 2. Clean monthly Sentinel-2 reflectance

Use the archive’s median of the clear passes for each field and season month. Preserve the ten reflectance bands and compute NDVI from the red and near-infrared bands. The reflectances are already scaled. A month without a clear pass stays empty. Each field uses the season calendar supplied with its survey.

In [4]:
bands = ["b02", "b03", "b04", "b05", "b06", "b07", "b08", "b8a", "b11", "b12"]
month_key = ["field_season_id", "season_month"]
year_key = [*month_key, "which_year"]
band_sql = ", ".join(bands)
calendar = db.sql("""
    SELECT field_season_id, season_month, which_year, year, month
    FROM monthly_clearest JOIN development USING (release, field_season_id)
""").df().set_index(year_key)
medians = db.sql(f"""
    SELECT field_season_id, season_month, which_year, n_clear, {band_sql}
    FROM monthly_median JOIN development USING (release, field_season_id)
    WHERE n_clear > 0
""").df().set_index(year_key)

assert calendar.index.is_unique and medians.index.is_unique

def normalized_difference(a, b):
    total = a + b
    return (a - b).div(total.where((total > 0) & (a >= 0) & (b >= 0)))

def by_month(frame, prefix):
    wide = frame.unstack("season_month").sort_index(axis=1)
    wide.columns = [f"{prefix}_{name}_m{month}" for name, month in wide.columns]
    return wide.reindex(fields.index)

current_calendar = calendar.xs("this_season", level="which_year")
current_median = medians.xs("this_season", level="which_year")
months = sorted(current_calendar.index.get_level_values("season_month").unique())

def optical_block(frame):
    values = frame[bands].replace([np.inf, -np.inf], np.nan).copy()
    values["ndvi"] = normalized_difference(values.b08, values.b04)
    return by_month(values, "opt")

median_opt = optical_block(current_median)
assert len(median_opt.columns) == len(months) * (len(bands) + 1)

## 3. Measure same-field, same-month change

Pair clear monthly medians only when both calendar months are known, the month numbers match and the years differ by exactly one. Compute current minus previous NDVI (greenness), NDMI (canopy moisture) and NDRE (red-edge contrast). A positive value means a larger index this season.

Divide each field’s scheduled months into early, middle and late thirds. Within a third, average the paired differences with weight equal to the smaller clear-pass count across the two years, capped at three. That gives a well-observed month more influence while limiting how much one month can dominate. This cap is a fixed design choice. Retain the number of valid NDVI pairs in each third, including zero. A field with no overlap has missing contrasts and still receives a prediction.

In [5]:
assert medians.index.isin(calendar.index).all()
dated = medians.join(calendar, validate="one_to_one")
current = dated.xs("this_season", level="which_year")
prior = dated.xs("previous_year", level="which_year")
pairs = current.join(prior, how="inner", lsuffix="_now", rsuffix="_prior", validate="one_to_one")
pairs = pairs.loc[(pairs.month_now == pairs.month_prior) &
                  (pairs.year_now == pairs.year_prior + 1)].copy()
season_size = current_calendar.groupby(level="field_season_id").size()
assert current_calendar.reset_index().groupby("field_season_id").season_month.max().eq(season_size).all()
pairs = pairs.reset_index()
relative = (pairs.season_month - 1) / pairs.field_season_id.map(season_size)
pairs["phase"] = pd.cut(relative, [0, 1/3, 2/3, 1], right=False, labels=["early", "mid", "late"])
weights = pairs[["n_clear_now", "n_clear_prior"]].min(axis=1).clip(upper=3)
group = [pairs.field_season_id, pairs.phase]
phase_blocks = {}
for index_name, (a, b) in {"ndvi": ("b08", "b04"), "ndmi": ("b08", "b11"),
                           "ndre": ("b8a", "b05")}.items():
    now = normalized_difference(pairs[a + "_now"], pairs[b + "_now"])
    prior_year = normalized_difference(pairs[a + "_prior"], pairs[b + "_prior"])
    delta = now - prior_year
    weight_sum = weights.where(delta.notna()).groupby(group, observed=True).sum(min_count=1)
    total = (delta * weights).groupby(group, observed=True).sum(min_count=1)
    phase_blocks["delta_" + index_name] = total.div(weight_sum).unstack()
    if index_name == "ndvi":
        counts = delta.groupby(group, observed=True).count().unstack()
        phase_blocks["paired_months"] = counts.reindex(fields.index).fillna(0).astype(int)

contrast = pd.concat(phase_blocks, axis=1).reindex(fields.index)
contrast.columns = ["_".join(col) for col in contrast.columns]
n_pairs = contrast.filter(like="paired_months_").sum(axis=1)
assert (n_pairs <= season_size.reindex(fields.index)).all()
coverage = fields[["country_iso3"]].assign(paired=n_pairs.gt(0), months=n_pairs)
print(coverage.groupby("country_iso3").agg(fields=("paired", "size"),
      pct_with_pair=("paired", lambda x: round(100*x.mean(), 1)),
      median_paired_months=("months", "median")).to_string())

              fields  pct_with_pair  median_paired_months
country_iso3                                             
BDI              831          100.0                   2.0
KEN            15315           96.4                   8.0
MWI             2011          100.0                   6.0
RWA              967           69.0                   2.0
TZA             1579          100.0                   5.0
UGA              196          100.0                   8.0
ZMB             1035          100.0                   5.0
ZWE             3645          100.0                   6.0


## 4. Retain environmental measurements with valid source flags

Keep valid monthly CHIRPS rainfall in mm and AgERA5 temperature in °C. Use one precipitation product throughout. This avoids mixing CHIRPS totals with AgERA5 precipitation records flagged for possible wraparound. Surface soil properties and terrain provide static context. The joins keep all development rows, and the export keeps nulls so each model can learn its own imputation values from training data.

In [6]:
weather = db.sql("""
    SELECT field_season_id, season_month,
           CASE WHEN chirps_valid AND chirps_precip_mm >= 0 THEN chirps_precip_mm END AS rain_mm,
           CASE WHEN agera5_temperature_valid THEN agera5_temperature_2m_k - 273.15 END AS temp_c
    FROM extra_rain_chirps FULL JOIN extra_weather_agera5
         USING (release, field_season_id, season_month, year, month)
    JOIN development USING (release, field_season_id)
""").df().set_index(month_key)
assert weather.index.is_unique
soil_names = ["clay", "sand", "phh2o", "soc", "nitrogen", "bdod", "cec"]
soil_sql = ", ".join(
    f"CASE WHEN soilgrids_valid THEN soilgrids_{name}_0_5cm_mean END AS soil_{name}_0_5cm"
    for name in soil_names
)
site = query_fields(f"""
    SELECT field_season_id, {soil_sql},
           CASE WHEN terrain_valid THEN elevation_m END AS elevation_m,
           CASE WHEN terrain_valid THEN slope_deg END AS slope_deg,
           CASE WHEN terrain_valid THEN relief_3x3_m END AS relief_3x3_m
    FROM development LEFT JOIN extra_soil USING (release, field_season_id)
                     LEFT JOIN extra_terrain USING (release, field_season_id)
""").reindex(fields.index)
context = pd.concat([by_month(weather, "wx"), site], axis=1)

## 5. Check coverage and export the clean table

The checks below show how many fields have any measurement from each source. Each source covers a different set of measurements. The historical coverage check counts observed changes, so a field with zero matched months is recorded as missing that source.

The export includes the target, field and survey identifiers, fold and buffer flags, and 140 predictors. Numeric predictors keep their missing values. `survey_year` is categorical. The Parquet round-trip check verifies every exported value and column type. **You can stop after this section to hand the dataset to the modeling team.**

In [7]:
survey = (fields.data_set + "/" + fields.season_year.astype(str)).rename("survey_year")
predictors = pd.concat([survey, median_opt, context, contrast], axis=1)
assert predictors.index.equals(fields.index) and predictors.columns.is_unique

feature_blocks = {
    "Current-season optical": median_opt,
    "Previous-year comparison": contrast.filter(like="delta_"),
    "Rain": context.filter(like="wx_rain_"),
    "Temperature": context.filter(like="wx_temp_"),
    "Topsoil": site.filter(like="soil_"),
    "Terrain": site[["elevation_m", "slope_deg", "relief_3x3_m"]],
}
coverage_rows = []
for name, block in feature_blocks.items():
    assert block.shape[1] > 0
    present = block.notna().any(axis=1)
    coverage_rows.append((name, present.sum(), (~present).sum()))
print(pd.DataFrame(coverage_rows, columns=["source", "any measurement", "all missing"])
      .set_index("source").to_string())

                          any measurement  all missing
source                                                
Current-season optical              25574            5
Previous-year comparison            24724          855
Rain                                25579            0
Temperature                         25579            0
Topsoil                             25273          306
Terrain                             25579            0


In [8]:
export = fields.join(predictors, validate="one_to_one").reset_index()
assert export.field_season_id.is_unique and len(export) == inventory.development.sum()
assert not np.isinf(export[predictors.columns.drop("survey_year")].to_numpy()).any()
out_path.parent.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(dir=out_path.parent) as stage:
    pending = Path(stage) / out_path.name
    export.to_parquet(pending, engine="pyarrow", compression="zstd", index=False)
    pd.testing.assert_frame_equal(export, pd.read_parquet(pending))
    pending.replace(out_path)
print(f"data/clean/{out_path.name} · {len(export):,} rows · "
      f"{len(predictors.columns)} predictors · {out_path.stat().st_size/1e6:.1f} MB")
notice = db.sql("SELECT text FROM readme ORDER BY line_no").df().text.to_list()
db.close()

data/clean/felix_seasonal_contrast.parquet · 25,579 rows · 140 predictors · 11.1 MB


### Data credits

The shared DuckDB release carries the following source notices. This method uses the yield labels, Sentinel-2 reflectance, CHIRPS, AgERA5, SoilGrids and Copernicus DEM.

In [9]:
start = next(i for i, line in enumerate(notice) if line.startswith("SOURCES"))
print("\n".join(notice[start + 1:]).strip())

Yields: One Acre Fund MEL data (CC BY) and its curated release (Zenodo 11122388, CC BY 4.0); Lacuna Fund / PULA
Zimbabwe (Zenodo 10060610, CC BY 4.0); Vision-guided crop yield data, Zimbabwe 2024 (Zenodo 17626117, CC BY 4.0); FAO
and NASA Harvest Malawi 2022, 2023 (CC BY). Contains modified Copernicus Sentinel data 2015-2024 (Earth Search,
Microsoft Planetary Computer). Copernicus DEM GLO-30; CHIRPS v2.0; SoilGrids 2.0 (ISRIC, CC BY 4.0); ERA5 and AgERA5
(Copernicus Climate Change Service; values modified: monthly sums, cell centres); Sentinel-1 RTC (Catalyst, Planetary
Computer, CC BY 4.0). The AlphaEarth Foundations Satellite Embedding dataset is produced by Google and Google
DeepMind. TESSERA v1.1 (University of Cambridge). Presto, Galileo (NASA Harvest). OlmoEarth (Ai2): no military or
extractive use; these limits pass on to anything made from it; cite Ai2.


## Optional validation · Ridge on the exported table

The comparison reloads the Parquet file and uses the team’s shared `train_test` function for outer and inner spatial folds. All four variants receive the same survey/year category and are evaluated on identical field IDs.

The reference resembles Florian’s single-pass method. It is rebuilt here on the published `keep` cohort, so its score compares feature choices within this experiment. Pamely’s notebook explores pixel-level quality thresholds. [Florian’s reference notebook](https://github.com/JinZida/DSC291DSSD/blob/cd3c5ee4834f4a86ff4c0ff77004bad7b328fbce/notebook/florian/01_cleaning_project_rules.ipynb) · [Pamely’s reference notebook](https://github.com/JinZida/DSC291DSSD/blob/0438d8d304237170bd6fc38ac40ef415a25680e2/notebook/Pamely/maize_cleaning.ipynb)

For outer fold *k*, test every unlocked row assigned to *k*. Remove those rows and their 2 km buffer from training. Inside that training set, repeat the fold and buffer rule for each remaining fold to select Ridge’s penalty from 10, 100 and 1,000 by mean validation MSE. The pipeline fits the median imputer, scaler and survey/year encoder separately on each training split. A feature missing throughout a training split contributes no learned slope. No coordinate, fold flag, identifier or target-derived field enters the feature matrix.

In [10]:
import sys

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

sys.path.insert(0, str(db_path.parents[1] / "scripts"))
from spatial_folds import train_test

ready = pd.read_parquet(out_path).set_index("field_season_id")
keys = ready.reset_index()[["release", "field_season_id"]]
with duckdb.connect(str(db_path), read_only=True) as check_db:
    check_db.register("exported_fields", keys)
    single_pass = check_db.sql(f"""
        SELECT field_season_id, season_month, {band_sql}
        FROM monthly_clearest JOIN exported_fields USING (release, field_season_id)
        WHERE which_year = 'this_season' AND clear
    """).df().set_index(month_key)
assert single_pass.index.is_unique
clear_opt = optical_block(single_pass).reindex(index=ready.index, columns=median_opt.columns)
proposed = ready[predictors.columns]
optical_columns = ["survey_year", *median_opt.columns]
arms = {
    "clearest": pd.concat([ready.survey_year, clear_opt], axis=1),
    "median": proposed[optical_columns],
    "median + context": proposed[[*optical_columns, *context.columns]],
    "median + context + change": proposed,
}
full_name = "median + context + change"
target = ready.yield_t_ha

In [11]:
def ridge_search(matrix, splits):
    numeric = matrix.columns.drop("survey_year").tolist()
    transform = ColumnTransformer([
        ("numeric", make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                                  StandardScaler()), numeric),
        ("survey", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["survey_year"]),
    ])
    return GridSearchCV(make_pipeline(transform, Ridge(solver="cholesky")),
                        {"ridge__alpha": [10, 100, 1000]}, cv=splits,
                        scoring="neg_mean_squared_error", error_score="raise")

predictions = pd.DataFrame(index=ready.index, columns=arms, dtype=float)
fold_log = []
with threadpool_limits(limits=2):
    for k in range(1, 5):
        train, test = train_test(ready, k)
        assert not (train & test).any() and not ready.loc[train, f"gap_{k}"].any()
        inner = ready.loc[train]
        splits = []
        for j in range(1, 5):
            if j != k:
                inner_train, inner_test = train_test(inner, j)
                splits.append((np.flatnonzero(inner_train), np.flatnonzero(inner_test)))
        assert all(len(a) and len(b) for a, b in splits)
        for name, matrix in arms.items():
            search = ridge_search(matrix, splits).fit(matrix.loc[train], target.loc[train])
            predictions.loc[test, name] = search.predict(matrix.loc[test])
            fold_log.append((k, int(train.sum()), int(test.sum()), name,
                             search.best_params_["ridge__alpha"]))
assert np.isfinite(predictions.to_numpy()).all()
fold_info = pd.DataFrame(fold_log, columns=["fold", "train", "test", "method", "alpha"])
print("Selected Ridge penalty by fold")
print(fold_info.pivot(index=["fold", "train", "test"], columns="method", values="alpha").to_string())

Selected Ridge penalty by fold
method           clearest  median  median + context  median + context + change
fold train test                                                               
1    17484 6271       100     100               100                        100
2    17132 6583       100     100               100                        100
3    17156 6361        10      10                10                         10
4    17370 6364        10      10               100                        100


### Ridge results by fold and source

Pooled RMSE and MAE weight each held-out field equally. Macro RMSE averages the source datasets’ RMSEs equally so Kenya’s larger sample does not dominate. Fold and source breakdowns show where changes help or hurt. Burundi and Uganda are evaluated as slices of the pooled model, with no independent country-specific fits.

In [12]:
def rmse(actual, predicted):
    return float(np.sqrt(mean_squared_error(actual, predicted)))

def grouped_rmse(by):
    return pd.DataFrame({
        group: {name: rmse(target.loc[idx], predictions.loc[idx, name]) for name in arms}
        for group, idx in by.groupby(by).groups.items()
    }).T.rename_axis(by.name)

by_source = grouped_rmse(ready.data_set)
summary = pd.DataFrame([
    {"method": name, "pooled_RMSE": rmse(target, pred), "MAE": mean_absolute_error(target, pred),
     "pooled_R2": r2_score(target, pred), "macro_source_RMSE": by_source[name].mean()}
    for name, pred in predictions.items()
]).set_index("method")
print("All out-of-fold predictions · errors in t/ha")
print(summary.round(4).to_string())
print("\nRMSE by outer fold")
print(grouped_rmse(ready.fold).round(4).to_string())
print("\nRMSE by source · negative change means lower error")
sources = by_source.assign(full_minus_clearest=by_source[full_name] - by_source.clearest)
sources.insert(0, "rows", ready.data_set.value_counts().reindex(sources.index))
print(sources.round(4).to_string())

All out-of-fold predictions · errors in t/ha
                           pooled_RMSE     MAE  pooled_R2  macro_source_RMSE
method                                                                      
clearest                        1.6037  1.2410     0.2122             1.5344
median                          1.6060  1.2434     0.2100             1.5274
median + context                1.5677  1.2125     0.2472             1.4841
median + context + change       1.5667  1.2110     0.2482             1.4816

RMSE by outer fold
      clearest  median  median + context  median + context + change
fold                                                               
1       1.5797  1.5717            1.5278                     1.5274
2       1.5869  1.5897            1.5576                     1.5546
3       1.6442  1.6571            1.6122                     1.6133
4       1.6036  1.6043            1.5720                     1.5701

RMSE by source · negative change means lower error
             

### What the comparison supports

The full method improved on the clearest-pass reference in all four spatial folds and eight of the eleven source datasets. Monthly medians alone slightly increased pooled RMSE. Adding environmental context then reduced it by 0.0383 t/ha. The historical contrast and coverage counts together added a small gain and improved three folds. In Rwanda, only 69% of fields supplied a historical pair. Burundi, Rwanda and the Zimbabwe Lacuna data had slightly higher error with the full method than with the reference.

These results support retaining the environmental measurements. The previous-year features are a modest, optional addition. The evaluation covers 22 eligible country-years of the 31 in the inventory. Predicting an unseen year or country would require a separate validation split.